In [1]:
import pandas as pd

df = pd.read_csv('data/measurements-out.csv')
df.head()  

: 

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 302052 entries, 0 to 302051
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Captured Time  302052 non-null  object 
 1   Latitude       302052 non-null  float64
 2   Longitude      302052 non-null  float64
 3   Value          302052 non-null  float64
 4   Unit           302052 non-null  object 
 5   Location Name  47504 non-null   object 
 6   Device ID      264021 non-null  float64
 7   MD5Sum         302052 non-null  object 
 8   Height         184856 non-null  float64
 9   Surface        0 non-null       float64
 10  Radiation      0 non-null       float64
 11  Uploaded Time  302052 non-null  object 
 12  Loader ID      36855 non-null   float64
dtypes: float64(8), object(5)
memory usage: 30.0+ MB


In [ ]:
df.count()

Captured Time    302052
Latitude         302052
Longitude        302052
Value            302052
Unit             302052
Location Name     47504
Device ID        264021
MD5Sum           302052
Height           184856
Surface               0
Radiation             0
Uploaded Time    302052
Loader ID         36855
dtype: int64

In [ ]:
df.describe()

,Latitude,Longitude,Value,Device ID,Height,Surface,Radiation,Loader ID
count,302052.000000,302052.000000,302052.000000,264021.000000,184856.000000,0.0,0.0,36855.000000
mean,39.878222,66.180903,44.417581,71693.250768,67.010846,NaN,NaN,60273.488292
std,6.608308,92.486511,228.252435,59223.557097,107.297483,NaN,NaN,11538.290631
min,0.000000,-123.075200,0.000000,74.000000,0.000000,NaN,NaN,32005.000000
25%,34.508338,5.875275,18.000000,238.000000,13.000000,NaN,NaN,46034.000000
50%,37.450012,136.153435,30.000000,65006.000000,34.000000,NaN,NaN,67251.000000
75%,42.565000,139.625711,42.000000,100242.000000,60.000000,NaN,NaN,67257.000000
max,58.475917,141.032974,7505.000000,330202.000000,569.000000,NaN,NaN,67276.000000


In [ ]:
df.isnull().sum()

Captured Time         0
Latitude              0
Longitude             0
Value                 0
Unit                  0
Location Name    254548
Device ID         38031
MD5Sum                0
Height           117196
Surface          302052
Radiation        302052
Uploaded Time         0
Loader ID        265197
dtype: int64

In [ ]:
invalid_lat = df['Latitude'].isna() | ~df['Latitude'].between(-90, 90)
invalid_lon = df['Longitude'].isna() | ~df['Longitude'].between(-180, 180)
invalid_coords = invalid_lat | invalid_lon
print(invalid_coords.sum())

0


In [ ]:
missing_device_id = df['Device ID'].isna() | (df['Device ID'].astype(str).str.strip() == '')
print(missing_device_id.sum())

38031


In [ ]:
missing_device_id = df['Uploaded Time'].isna() | (df['Uploaded Time'].astype(str).str.strip() == '')
print(missing_device_id.sum())

0


In [ ]:
matches = df['Captured Time'].astype(str).str.match(r'^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$')
invalid = df[~matches]
print(f"Rows with invalid format: {len(invalid)}")
df=df[matches]
len(f"Rows after deleting invalid format: {len(invalid)}")

Rows with invalid format: 23833


41

In [ ]:
import pandas as pd

# Extract only minutes and seconds from 'Captured Time'
df['Captured_MinSec'] = (
    df['Captured Time'].dt.minute * 60 +
    df['Captured Time'].dt.second
)
df['Captured_MinSec'] = pd.to_timedelta(df['Captured_MinSec'], unit='s')

# Calculate raw diff
df['time_diff'] = (df['Captured_MinSec'] - df['Uploaded Time']).abs()

# Wrap-around logic: if diff > 30 min, take 60 min - diff
wrap_limit = pd.Timedelta(minutes=30)
df['time_diff'] = df['time_diff'].apply(
    lambda x: min(x, pd.Timedelta(minutes=60) - x)
)

# Max diff
max_diff = df['time_diff'].max()

print(f"Max time difference (minutes+seconds only, wrap handled): {max_diff}")
df[df['time_diff'] == max_diff]


Max time difference (minutes+seconds only, wrap handled): 0 days 00:29:58.700000


,Captured Time,Latitude,Longitude,Value,Unit,Location Name,Device ID,MD5Sum,Height,Surface,Radiation,Uploaded Time,Loader ID,Captured_Time_only,time_diff,Captured_MinSec
35309,2018-07-17 10:24:00,36.810847,127.164638,26.0,cpm,NaN,NaN,6953bd9ebb8d8f8efc93cf1c0f288ab3,NaN,NaN,NaN,0 days 00:54:01.300000,37029.0,0 days 10:24:00,0 days 00:29:58.700000,0 days 00:24:00
35310,2018-07-17 10:24:00,36.810838,127.164643,30.0,cpm,NaN,NaN,7f141f2963c80617b6bfb1b26f95b2ad,NaN,NaN,NaN,0 days 00:54:01.300000,37029.0,0 days 10:24:00,0 days 00:29:58.700000,0 days 00:24:00
35311,2018-07-17 10:24:00,36.810832,127.164648,25.0,cpm,NaN,NaN,7fd3d7433f11e0cbe7b4bba686deddcb,NaN,NaN,NaN,0 days 00:54:01.300000,37029.0,0 days 10:24:00,0 days 00:29:58.700000,0 days 00:24:00
35312,2018-07-17 10:24:00,36.810823,127.164647,26.0,cpm,NaN,NaN,97b7693ae258af1cca6d146cd6ccbad8,NaN,NaN,NaN,0 days 00:54:01.300000,37029.0,0 days 10:24:00,0 days 00:29:58.700000,0 days 00:24:00
35313,2018-07-17 10:24:00,36.810815,127.164635,24.0,cpm,NaN,NaN,faf5c36edd7a98e790a2beb6ece15d30,NaN,NaN,NaN,0 days 00:54:01.300000,37029.0,0 days 10:24:00,0 days 00:29:58.700000,0 days 00:24:00
35314,2018-07-17 10:24:00,36.810805,127.164625,26.0,cpm,NaN,NaN,4384ff0f40f89d1a2acc9a15fa288c21,NaN,NaN,NaN,0 days 00:54:01.300000,37029.0,0 days 10:24:00,0 days 00:29:58.700000,0 days 00:24:00
35315,2018-07-17 10:24:00,36.810807,127.164623,28.0,cpm,NaN,NaN,f8ac3670630f586b64c69cb19209364c,NaN,NaN,NaN,0 days 00:54:01.300000,37029.0,0 days 10:24:00,0 days 00:29:58.700000,0 days 00:24:00
35316,2018-07-17 10:24:00,36.810807,127.164622,30.0,cpm,NaN,NaN,0a61063680190c8183a2fff0a3356e31,NaN,NaN,NaN,0 days 00:54:01.300000,37029.0,0 days 10:24:00,0 days 00:29:58.700000,0 days 00:24:00
35317,2018-07-17 10:24:00,36.810805,127.164623,31.0,cpm,NaN,NaN,6c5af030fdaa2bd1b3e00bc07491ce65,NaN,NaN,NaN,0 days 00:54:01.300000,37029.0,0 days 10:24:00,0 days 00:29:58.700000,0 days 00:24:00
35318,2018-07-17 10:24:00,36.810805,127.164622,31.0,cpm,NaN,NaN,e784cb9275240621a849dc5c36ecbdd3,NaN,NaN,NaN,0 days 00:54:01.300000,37029.0,0 days 10:24:00,0 days 00:29:58.700000,0 days 00:24:00


In [ ]:
import pandas as pd

chunk = pd.read_csv('data/measurements-out.csv', nrows=100000)

chunk.to_csv('data/test.csv', index=False)